# 04 Error analysis

Where does the chosen tree model go wrong? Everything is computed on the test set of each split from the fitted
candidates saved by `train`, and every sentence in the summary is generated from those numbers.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from house_prices import clean, config, load

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")
config.PATHS.figures_dir.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.savefig(config.PATHS.figures_dir / f"{name}.png", dpi=140, bbox_inches="tight")

In [ ]:
import json

import joblib
from IPython.display import display

from house_prices import evaluate, features, formatting, split

metrics = json.loads(config.PATHS.metrics_json.read_text())
frames = {}
for setup in split.SETUPS:
    train_df, test_df = split.load_split(setup)
    chosen = metrics["setups"][setup]["chosen_model"]
    model = joblib.load(config.PATHS.candidates_dir / f"{setup}_{chosen}.joblib")
    frames[setup] = evaluate.build_error_frame(test_df, train_df, model.predict(features.make_inputs(test_df)))
    print(setup, evaluate.MODEL_LABELS[chosen], f"{len(test_df):,} test listings")

## Predicted against actual, and residuals

In [ ]:
for setup, frame in frames.items():
    label = evaluate.MODEL_LABELS[metrics["setups"][setup]["chosen_model"]]
    fig = evaluate.plot_predicted_vs_actual(frame, setup, label)
    plt.show()
    fig = evaluate.plot_residuals(frame, setup, label)
    plt.show()

## Error by city, property type, price decile and location coverage

In [ ]:
for setup, frame in frames.items():
    analysis = metrics["setups"][setup]["error_analysis"]
    label = evaluate.MODEL_LABELS[metrics["setups"][setup]["chosen_model"]]
    fig = evaluate.plot_error_by_group(analysis, setup, label)
    plt.show()
    for key in ("by_city", "by_property_type", "by_price_decile", "by_location_listings"):
        table = pd.DataFrame(analysis[key])
        table["mdape"] = (table["mdape"] * 100).round(1)
        table["bias_pct"] = (table["bias_pct"] * 100).round(1)
        table["mae_pkr"] = table["mae_pkr"].round(0)
        print(f"{setup} split: {key}")
        display(table)

## Bias by price decile

Positive values mean the model over-predicts that band on average; negative means it under-predicts. Trees pull
extreme prices toward the middle, so the cheapest deciles tend to be over-predicted and the most expensive
under-predicted. The table shows whether that happens here.

In [ ]:
for setup, frame in frames.items():
    decile = frame.groupby("price_decile").agg(listings=("ape", "size"), median_error=("ape", "median"), bias=("log_ratio", lambda v: np.exp(v.median()) - 1))
    fig, ax = plt.subplots(figsize=(7, 3.4))
    sns.barplot(x=decile.index, y=decile["bias"] * 100, ax=ax, color="#4c78a8")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xlabel("Price decile (1 = cheapest)")
    ax.set_ylabel("Median bias (%)")
    ax.set_title(f"Median over/under-prediction by price decile ({setup} split)")
    plt.show()

## The ten worst predictions

In [ ]:
for setup in split.SETUPS:
    worst = pd.DataFrame(metrics["setups"][setup]["error_analysis"]["worst_10"])
    show = worst.assign(actual=[formatting.format_pkr(v) for v in worst["actual_price"]], predicted=[formatting.format_pkr(v) for v in worst["predicted_price"]], error=(worst["ape"] * 100).round(0).astype(int).astype(str) + "%")
    print(f"{setup} split")
    display(show[["city", "location", "property_type", "area_sqft", "bedrooms", "baths", "actual", "predicted", "error", "location_train_listings", "likely_reason"]])

The reasons are automatic guesses from simple rules (price per sq ft against the location's usual level,
area per bedroom, and location coverage). They point at likely data entry problems; they are not verified.

## Summary of the weakest areas

In [ ]:
for setup in split.SETUPS:
    print(f"{setup} split")
    for line in metrics["setups"][setup]["error_analysis"]["weakest"]:
        print("-", line)
    print()

## What the model relies on (SHAP)

In [ ]:
from PIL import Image

for name in ("shap_bar", "shap_beeswarm", "shap_waterfall_example"):
    path = config.PATHS.figures_dir / f"{name}.png"
    if path.exists():
        plt.figure(figsize=(9, 5.5))
        plt.imshow(Image.open(path))
        plt.axis("off")
        plt.show()
shap_block = metrics.get("shap")
if shap_block:
    print(shap_block["example_sentence"])
    display(pd.DataFrame(shap_block["group_importance"]).round(4))